In [1]:
import random
import heapq
from collections import defaultdict
import matplotlib.pyplot as plt
import networkx as nx

In [2]:
# === Union-Find Data Structure ===
class UnionFind:
    def _init_(self, n):
        self.parent = list(range(n))

    def find(self, u):
        while self.parent[u] != u:
            self.parent[u] = self.parent[self.parent[u]]
            u = self.parent[u]
        return u

    def union(self, u, v):
        pu, pv = self.find(u), self.find(v)
        if pu != pv:
            self.parent[pu] = pv
            return True
        return False

In [3]:
# === Graph Algorithms ===

# Kruskal's Algorithm
def kruskal(n, edges):
    edges.sort(key=lambda x: x[2])
    uf = UnionFind(n)
    mst = []
    total_weight = 0
    for u, v, w in edges:
        if uf.union(u, v):
            mst.append((u, v, w))
            total_weight += w
    return mst, total_weight

In [4]:
# Prim's Algorithm
def prim(n, edges, start=1):
    adj = defaultdict(list)
    for u, v, w in edges:
        adj[u].append((w, v))
        adj[v].append((w, u))

    visited = [False] * n
    min_heap = [(0, start)]
    total_weight = 0
    mst = []

    while min_heap:
        w, u = heapq.heappop(min_heap)
        if visited[u]:
            continue
        visited[u] = True
        total_weight += w
        for next_w, v in adj[u]:
            if not visited[v]:
                heapq.heappush(min_heap, (next_w, v))
                mst.append((u, v, next_w))
    return mst, total_weight

In [5]:
# Boruvka's Algorithm (Safe Version)
def boruvka_safe(n, edges):
    uf = UnionFind(n)
    mst = []
    total_weight = 0
    num_components = n

    while num_components > 1:
        cheapest = [-1] * n
        changed = False

        for i, (u, v, w) in enumerate(edges):
            set_u = uf.find(u)
            set_v = uf.find(v)
            if set_u != set_v:
                if cheapest[set_u] == -1 or edges[cheapest[set_u]][2] > w:
                    cheapest[set_u] = i
                if cheapest[set_v] == -1 or edges[cheapest[set_v]][2] > w:
                    cheapest[set_v] = i

        for i in range(n):
            if cheapest[i] != -1:
                u, v, w = edges[cheapest[i]]
                if uf.union(u, v):
                    mst.append((u, v, w))
                    total_weight += w
                    num_components -= 1
                    changed = True

        if not changed:
            break

    return mst, total_weight

In [6]:
# Reverse Delete Algorithm
def reverse_delete(n, edges):
    edges_sorted = sorted(edges, key=lambda x: -x[2])
    mst = edges[:]

    def is_connected(subgraph_edges):
        uf = UnionFind(n)
        for u, v, _ in subgraph_edges:
            uf.union(u, v)
        root = uf.find(0)
        return all(uf.find(i) == root for i in range(n) if any(i in e[:2] for e in subgraph_edges))

    for u, v, w in edges_sorted:
        mst.remove((u, v, w))
        if not is_connected(mst):
            mst.append((u, v, w))

    total_weight = sum(w for _, _, w in mst)
    return mst, total_weight

In [7]:
# Karger's Min-Cut Algorithm
def karger_min_cut(n, edges, iterations=20):
    min_cut = float('inf')

    for _ in range(iterations):
        parent = list(range(n))
        e = edges[:]

        def find(u):
            while parent[u] != u:
                parent[u] = parent[parent[u]]
                u = parent[u]
            return u

        def union(u, v):
            pu, pv = find(u), find(v)
            if pu != pv:
                parent[pu] = pv

        vertices = n
        while vertices > 2 and e:
            u, v, _ = random.choice(e)
            if find(u) != find(v):
                union(u, v)
                vertices -= 1
            e = [edge for edge in e if find(edge[0]) != find(edge[1])]

        cut = len([edge for edge in edges if find(edge[0]) != find(edge[1])])
        min_cut = min(min_cut, cut)

    return min_cut


In [8]:
# === Visualization Function ===
def visualize_graph(edges, mst_edges=None, title="Graph"):
    G = nx.Graph()
    for u, v, w in edges:
        G.add_edge(u, v, weight=w)

    pos = nx.spring_layout(G, seed=42)
    edge_labels = nx.get_edge_attributes(G, 'weight')

    plt.figure(figsize=(8, 6))
    nx.draw(G, pos, with_labels=True, node_color='lightblue', edge_color='gray', node_size=500)
    nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels)

    if mst_edges:
        mst_G = nx.Graph()
        mst_G.add_edges_from([(u, v) for u, v, _ in mst_edges])
        nx.draw(mst_G, pos, with_labels=False, edge_color='green', width=2)

    plt.title(title)
    plt.show()

In [9]:
# === Example: Using ENZYMES_g237 Data ===
# Example parsed graph
edges_237 = [
    (2, 4, 1), (1, 2, 1), (1, 5, 1), (4, 6, 1), (1, 4, 1),
    (2, 6, 1), (5, 6, 1), (3, 6, 1), (1, 6, 1), (1, 3, 1), (3, 5, 1)
]
num_nodes_237 = 7  # max node index + 1


In [10]:
# Run all algorithms
kruskal_mst, _ = kruskal(num_nodes_237, edges_237)
prim_mst, _ = prim(num_nodes_237, edges_237, start=1)
boruvka_mst, _ = boruvka_safe(num_nodes_237, edges_237)
reverse_delete_mst, _ = reverse_delete(num_nodes_237, edges_237)
karger_cut = karger_min_cut(num_nodes_237, edges_237)


TypeError: UnionFind() takes no arguments

In [ ]:
# Visualize everything
visualize_graph(edges_237, title="Original Graph")
visualize_graph(edges_237, kruskal_mst, title="Kruskal's MST")
visualize_graph(edges_237, prim_mst, title="Prim's MST")
visualize_graph(edges_237, boruvka_mst, title="Borůvka's MST")
visualize_graph(edges_237, reverse_delete_mst, title="Reverse Delete Result")

# Print cut result
print("Karger's Min-Cut Estimate:", karger_cut)